# Stochastic-Driven Deep Learning Models for Financial Recommendation Systems
## Borsa Istanbul (BIST30) Application — YKBNK.IS

**Kadir Has University — FENS 402-S05 Engineering Design Project**

---

### Project Pipeline
1. Data Collection & Preprocessing (YKBNK.IS, 2020–2025)
2. Feature Engineering  
   - Technical Indicators (EMA20/50, RSI, Log Return, Volatility, Range)
   - Monte Carlo Simulation Outputs (MC_Expected, MC_Volatility, MC_VaR95)
   - Hidden Markov Model Regime Features (HMM_State, HMM_Prob_0/1, State_Change)
3. LSTM Model Training & Evaluation
4. Feature Contribution Analysis (Feature Elimination, SHAP, Permutation Importance, Integrated Gradients)
5. Final Optimized Model (Selected Features)
6. Crossover Trading Strategy Backtest

---
### Setup
Before running, install dependencies:  
```bash
pip install -r requirements.txt
```

## Cell 1 — Imports & Configuration

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy

import yfinance as yf
from hmmlearn import hmm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

import shap
from xgboost import XGBRegressor

# -- reproducibility ----------------------------------------------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -- configuration ------------------------------------------------------------
SYMBOL     = 'YKBNK.IS'
START_DATE = '2020-01-01'
END_DATE   = '2025-05-17'
SEQ_LEN    = 10       # LSTM look-back window (days)
N_HMM      = 2        # number of HMM hidden states
MC_WINDOW  = 60       # rolling window for Monte Carlo
MC_SIMS    = 200      # Monte Carlo simulation paths
MC_DAYS    = 10       # forward simulation horizon

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'Pandas version     : {pd.__version__}')
print('All imports successful.')

: 

## Cell 2 — Data Loading & Preprocessing

Downloads OHLCV data for `YKBNK.IS` from Yahoo Finance (2020–2025) and performs basic cleaning.

In [ ]:
# -- download -----------------------------------------------------------------
raw = yf.download(SYMBOL, start=START_DATE, end=END_DATE, progress=False)

# yfinance may return multi-level columns (Ticker, Field) - flatten them
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.dropna(inplace=True)
df.index = pd.to_datetime(df.index)

print(f'Downloaded {len(df)} trading days for {SYMBOL}')
print(f'Date range: {df.index[0].date()} -> {df.index[-1].date()}')
df.tail(3)

## Cell 3 — Technical Indicator Feature Engineering

Computed entirely with NumPy/Pandas — **no TA-Lib required**.

| Feature | Description |
|---------|-------------|
| `log_return` | ln(Close_t / Close_{t-1}) |
| `RSI` | Relative Strength Index (14-day) |
| `EMA20` / `EMA50` | Exponential Moving Averages |
| `EMA_diff` | EMA20 − EMA50 (trend signal) |
| `Volatility` | Rolling 10-day std of log returns |
| `Log` | ln(Close) |
| `Returns` | pct_change of Log |
| `Range` | (High / Low) − 1 |

In [ ]:
# -- helper: RSI (pure pandas, no TA-Lib) -------------------------------------
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

# -- helper: EMA (pure pandas) ------------------------------------------------
def compute_ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False, min_periods=period).mean()

# -- compute indicators -------------------------------------------------------
close = df['Close'].squeeze()   # ensure 1-D Series

df['log_return'] = np.log(close / close.shift(1))
df['RSI']        = compute_rsi(close, 14)
df['EMA20']      = compute_ema(close, 20)
df['EMA50']      = compute_ema(close, 50)
df['EMA_diff']   = df['EMA20'] - df['EMA50']
df['Volatility'] = df['log_return'].rolling(window=10).std()
df['Log']        = np.log(close)
df['Returns']    = df['Log'].pct_change()
df['Range']      = (df['High'] / df['Low']) - 1

df.dropna(inplace=True)
print(f'Rows after technical indicator drop-NaN: {len(df)}')
print('Features so far:', df.columns.tolist())

## Cell 4 — Monte Carlo Simulation Features

Uses Geometric Brownian Motion (GBM) over a rolling 60-day window to generate:
- **MC_Expected** — mean simulated price at horizon
- **MC_Volatility** — std of simulated prices
- **MC_VaR95** — 5th-percentile price (95 % Value-at-Risk)

In [ ]:
def monte_carlo_features(
    df: pd.DataFrame,
    window: int = MC_WINDOW,
    num_simulations: int = MC_SIMS,
    days: int = MC_DAYS
) -> pd.DataFrame:
    """
    For each row i, simulate `num_simulations` GBM paths of length `days`
    from the last `window` days of log-returns and extract summary statistics.
    """
    expected_prices, volatilities, var_95s = [], [], []

    close_series = df['Close'].squeeze()

    for i in range(len(df)):
        if i < window:
            expected_prices.append(np.nan)
            volatilities.append(np.nan)
            var_95s.append(np.nan)
            continue

        window_slice = close_series.iloc[i - window:i]
        log_returns  = np.log(1 + window_slice.pct_change().dropna())
        last_price   = float(window_slice.iloc[-1])

        rng = np.random.default_rng(seed=i)   # per-window seed for reproducibility
        walks = rng.normal(log_returns.mean(), log_returns.std(), (num_simulations, days))
        simulations = last_price * np.exp(np.cumsum(walks, axis=1))

        expected_prices.append(simulations[:, -1].mean())
        volatilities.append(simulations[:, -1].std())
        var_95s.append(np.percentile(simulations[:, -1], 5))

    df = df.copy()
    df['MC_Expected']  = expected_prices
    df['MC_Volatility'] = volatilities
    df['MC_VaR95']     = var_95s
    return df

print('Running Monte Carlo simulation ... (this may take ~30 s)')
df = monte_carlo_features(df)
df.dropna(inplace=True)
print(f'Rows after Monte Carlo drop-NaN: {len(df)}')

## Cell 5 — Hidden Markov Model (HMM) Regime Features

A 2-state Gaussian HMM is fitted on the 5 base OHLCV columns to discover latent
market regimes (e.g. bull vs. bear).  The following features are extracted:

| Feature | Description |
|---------|-------------|
| `HMM_State` | Most-likely regime label (0 or 1) |
| `HMM_Prob_0` | Posterior probability of being in state 0 |
| `HMM_Prob_1` | Posterior probability of being in state 1 |
| `State_Change` | 1 if regime changed from prior day, else 0 |

In [ ]:
# -- fit HMM on OHLCV ---------------------------------------------------------
hmm_obs   = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
hmm_model = hmm.GaussianHMM(n_components=N_HMM, n_iter=100, random_state=SEED)
hmm_model.fit(hmm_obs)

# -- extract features ---------------------------------------------------------
obs = df.copy()
obs['HMM_State']  = hmm_model.predict(hmm_obs)
proba = hmm_model.predict_proba(hmm_obs)
obs['HMM_Prob_0'] = proba[:, 0]
obs['HMM_Prob_1'] = proba[:, 1]

obs['Prev_HMM_State'] = obs['HMM_State'].shift(1)
obs['State_Change']   = (obs['HMM_State'] != obs['Prev_HMM_State']).astype(int)
obs.drop(columns=['Prev_HMM_State'], inplace=True)

obs.replace([np.inf, -np.inf], np.nan, inplace=True)
obs.dropna(inplace=True)

print(f'Final feature set has {obs.shape[1]} columns and {obs.shape[0]} rows.')
print('Columns:', obs.columns.tolist())

## Cell 6 — Normalisation & LSTM Dataset Construction

In [ ]:
# -- MinMax scale the full feature matrix -------------------------------------
scaler = MinMaxScaler()
scaled = scaler.fit_transform(obs)

# -- CLOSE column index (needed for inverse-transform later) ------------------
CLOSE_IDX = obs.columns.tolist().index('Close')
print(f"'Close' column index in scaled array: {CLOSE_IDX}")

# -- sliding-window LSTM sequences --------------------------------------------
def create_lstm_data(data: np.ndarray, seq_len: int = SEQ_LEN):
    """
    Build supervised (X, y) pairs:
      X[i] = data[i : i+seq_len]        shape (seq_len, n_features)
      y[i] = data[i+seq_len][CLOSE_IDX] scalar - next-day Close (scaled)
    """
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len][CLOSE_IDX])
    return np.array(X), np.array(y)

X, y = create_lstm_data(scaled, SEQ_LEN)
print(f'X shape: {X.shape}  |  y shape: {y.shape}')

## Cell 7 — LSTM Model Training

Architecture: single LSTM(64) → Dense(1), trained with Adam + MSE loss.
EarlyStopping on training loss (patience = 3).

In [ ]:
early_stop = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

model_lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(X.shape[1], X.shape[2])),
    Dense(1)
], name='LSTM_HMM_Model')

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.summary()

history = model_lstm.fit(
    X, y,
    epochs=100,
    batch_size=16,
    verbose=1,
    callbacks=[early_stop]
)

# -- training loss curve ------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'])
plt.title('LSTM Training Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 8 — Predictions & Performance Metrics

In [ ]:
# -- generate predictions -----------------------------------------------------
predicted_scaled = model_lstm.predict(X, verbose=0)

# inverse-transform: rebuild a dummy full-width array, place Close predictions
pred_full = np.zeros((len(predicted_scaled), scaled.shape[1]))
pred_full[:, CLOSE_IDX] = predicted_scaled[:, 0]
restored = scaler.inverse_transform(pred_full)[:, CLOSE_IDX]

# aligned actual prices
real_close = obs['Close'].iloc[SEQ_LEN:SEQ_LEN + len(restored)].values
dates      = obs.index[SEQ_LEN:SEQ_LEN + len(restored)]

# -- metrics ------------------------------------------------------------------
mae  = mean_absolute_error(real_close, restored)
mse  = mean_squared_error(real_close, restored)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(real_close, restored) * 100

print('=' * 40)
print('   LSTM + HMM - Full Dataset Results')
print('=' * 40)
print(f'  MAE  : {mae:.4f}')
print(f'  MSE  : {mse:.4f}')
print(f'  RMSE : {rmse:.4f}')
print(f'  MAPE : {mape:.2f} %')
print('=' * 40)

## Cell 9 — Visualisations

In [ ]:
# -- full historical forecast --------------------------------------------------
plt.figure(figsize=(14, 6))
plt.plot(dates, real_close, label='Actual Close', linewidth=1.5)
plt.plot(dates, restored,   label='LSTM + HMM Prediction', linewidth=1.5, alpha=0.8)
plt.title(f'{SYMBOL} - Full Historical Price Forecast (LSTM + HMM)')
plt.xlabel('Date')
plt.ylabel('Price (TRY)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# -- last 100 days -------------------------------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(dates[-100:], real_close[-100:], marker='o', label='Actual Close', linewidth=2)
plt.plot(dates[-100:], restored[-100:],   marker='x', linestyle='--',
         label='Predicted Close', linewidth=2)
plt.title(f'{SYMBOL} - Last 100 Days: Actual vs LSTM + HMM Prediction')
plt.xlabel('Date')
plt.ylabel('Price (TRY)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 10 — Next-Day Price Prediction

In [ ]:
last_sequence  = np.expand_dims(scaled[-SEQ_LEN:], axis=0)   # shape (1, 10, n_features)
next_scaled    = model_lstm.predict(last_sequence, verbose=0)

dummy = np.zeros((1, scaled.shape[1]))
dummy[0, CLOSE_IDX] = next_scaled[0, 0]
next_day_price = scaler.inverse_transform(dummy)[0, CLOSE_IDX]

last_known_date = obs.index[-1]
print(f'Last known date    : {last_known_date.date()}')
print(f'Last known Close   : {float(obs["Close"].iloc[-1]):.2f} TRY')
print(f'Predicted next day : {next_day_price:.2f} TRY')

---
## Cell 11 — Feature Contribution Analysis: Feature Elimination

Train a fresh model with each feature removed in turn and compare RMSE.
Features that cause the largest RMSE increase when removed are the most important.

In [ ]:
feature_names = [str(c) for c in obs.columns.tolist()]

def evaluate_without_feature(
    X: np.ndarray,
    y: np.ndarray,
    scaler: MinMaxScaler,
    removed_idx: int | None = None,
    epochs: int = 20
) -> tuple:
    X_mod = np.delete(X, removed_idx, axis=2) if removed_idx is not None else X.copy()

    model = Sequential([
        LSTM(64, activation='relu', input_shape=(X_mod.shape[1], X_mod.shape[2])),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
    model.fit(X_mod, y, epochs=epochs, batch_size=16, verbose=0, callbacks=[es])

    preds = model.predict(X_mod, verbose=0)
    pf    = np.zeros((len(preds), scaler.n_features_in_))
    pf[:, CLOSE_IDX] = preds[:, 0]
    rest  = scaler.inverse_transform(pf)[:, CLOSE_IDX]

    actual = obs['Close'].iloc[SEQ_LEN:SEQ_LEN + len(rest)].values
    mae_v  = mean_absolute_error(actual, rest)
    mse_v  = mean_squared_error(actual, rest)
    rmse_v = np.sqrt(mse_v)
    mape_v = mean_absolute_percentage_error(actual, rest) * 100
    return mae_v, mse_v, rmse_v, mape_v

print(f'Running feature elimination for {len(feature_names)} features ...')
elim_results = []
for i, fname in enumerate(feature_names):
    print(f'  [{i+1}/{len(feature_names)}] removing "{fname}"')
    mae_v, mse_v, rmse_v, mape_v = evaluate_without_feature(X, y, scaler, removed_idx=i)
    elim_results.append((fname, mae_v, mse_v, rmse_v, mape_v))

results_df = pd.DataFrame(elim_results, columns=['Feature', 'MAE', 'MSE', 'RMSE', 'MAPE'])
results_df.sort_values('RMSE', ascending=True, inplace=True)
results_df.reset_index(drop=True, inplace=True)
print('\nFeature Elimination Results (sorted by RMSE):')
print(results_df.to_string(index=False))

## Cell 12 — Feature Elimination Plots

In [ ]:
metrics = [('RMSE', 'steelblue'), ('MAE', 'coral'), ('MSE', 'mediumseagreen'), ('MAPE', 'mediumpurple')]

for metric, color in metrics:
    plt.figure(figsize=(10, 6))
    plt.barh(results_df['Feature'], results_df[metric], color=color)
    plt.xlabel(f'{metric} (when feature is removed)')
    plt.ylabel('Feature')
    plt.title(f'{metric} Value When Each Feature Is Removed')
    plt.gca().invert_yaxis()
    plt.grid(True, axis='x')
    plt.tight_layout()
    plt.show()

## Cell 13 — SHAP Feature Importance (via XGBoost surrogate)

In [ ]:
X_shap = obs.copy()
y_shap = X_shap.pop('Close')

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_shap, y_shap, test_size=0.2, random_state=SEED
)

xgb_model = XGBRegressor(n_estimators=100, random_state=SEED, verbosity=0)
xgb_model.fit(X_train_s, y_train_s)

explainer   = shap.Explainer(xgb_model)
shap_values = explainer(X_test_s)

plt.title('SHAP Summary Plot - Feature Importance (XGBoost surrogate)')
shap.summary_plot(shap_values, X_test_s, show=True)

## Cell 14 — Permutation Importance (LSTM Model)

In [ ]:
def permutation_importance_lstm(
    model,
    X: np.ndarray,
    scaler: MinMaxScaler,
    feature_names: list
) -> list:
    """Compute MSE increase when each feature is randomly shuffled."""
    # baseline
    base_preds = model.predict(X, verbose=0)
    base_full  = np.zeros((len(base_preds), scaler.n_features_in_))
    base_full[:, CLOSE_IDX] = base_preds[:, 0]
    base_rest  = scaler.inverse_transform(base_full)[:, CLOSE_IDX]
    actual     = obs['Close'].iloc[SEQ_LEN:SEQ_LEN + len(base_rest)].values
    base_score = mean_squared_error(actual, base_rest)

    importances = []
    rng = np.random.default_rng(SEED)
    for i in range(X.shape[2]):
        X_perm = X.copy()
        rng.shuffle(X_perm[:, :, i])   # shuffle along sample axis only
        preds  = model.predict(X_perm, verbose=0)
        pf     = np.zeros((len(preds), scaler.n_features_in_))
        pf[:, CLOSE_IDX] = preds[:, 0]
        rest   = scaler.inverse_transform(pf)[:, CLOSE_IDX]
        score  = mean_squared_error(actual, rest)
        importances.append(score - base_score)
    return importances

feat_names_clean = [str(c) for c in obs.columns]
importances      = permutation_importance_lstm(model_lstm, X, scaler, feat_names_clean)

imp_df = pd.DataFrame({'Feature': feat_names_clean, 'Importance': importances})
imp_df.sort_values('Importance', ascending=False, inplace=True)

plt.figure(figsize=(10, 6))
plt.barh(imp_df['Feature'], imp_df['Importance'], color='teal')
plt.xlabel('Increase in MSE (when feature is permuted)')
plt.title('Permutation Importance - LSTM Model')
plt.gca().invert_yaxis()
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

## Cell 15 — Integrated Gradients (LSTM Attribution)

In [ ]:
def integrated_gradients(
    model,
    input_tensor: tf.Tensor,
    baseline: tf.Tensor,
    steps: int = 50
) -> np.ndarray:
    """Approximate Integrated Gradients attributions for a single input."""
    alphas = tf.linspace(0.0, 1.0, steps + 1)          # (steps+1,)
    interp = baseline + alphas[:, None, None] * (input_tensor - baseline)  # (steps+1, seq, feat)

    with tf.GradientTape() as tape:
        tape.watch(interp)
        preds = model(interp)          # (steps+1, 1)

    grads   = tape.gradient(preds, interp).numpy()       # (steps+1, seq, feat)
    avg_g   = np.mean(grads[:-1], axis=0)                # (seq, feat)
    delta   = (input_tensor - baseline).numpy()          # (seq, feat)
    return delta * avg_g                                  # (seq, feat)

input_tensor = tf.constant(X[-1:], dtype=tf.float32)    # last sample
baseline     = tf.zeros_like(input_tensor)

ig_attr  = integrated_gradients(model_lstm, input_tensor[0], baseline[0])
avg_ig   = np.mean(ig_attr, axis=0)   # average over time steps -> (n_features,)

plt.figure(figsize=(10, 6))
plt.barh(feat_names_clean, avg_ig, color='darkorange')
plt.xlabel('Integrated Gradient Attribution')
plt.title('Feature Contribution to LSTM Prediction (Integrated Gradients)')
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

---
## Cell 16 — Final Optimised LSTM Model (Selected Features)

Using the top-10 features identified by the contribution analyses:
`Close`, `HMM_Prob_1`, `Low`, `High`, `Log`, `RSI`, `HMM_Prob_0`, `MC_VaR95`, `log_return`, `HMM_State`

In [ ]:
SELECTED_FEATURES = [
    'Close', 'HMM_Prob_1', 'Low', 'High', 'Log',
    'RSI', 'HMM_Prob_0', 'MC_VaR95', 'log_return', 'HMM_State'
]

# ensure all selected features exist
available = [f for f in SELECTED_FEATURES if f in obs.columns]
missing   = set(SELECTED_FEATURES) - set(available)
if missing:
    print(f'Warning: features not found and skipped: {missing}')
SELECTED_FEATURES = available

df_sel  = obs[SELECTED_FEATURES].copy().dropna()
scaler2 = MinMaxScaler()
scaled2 = scaler2.fit_transform(df_sel)

CLOSE_IDX2 = SELECTED_FEATURES.index('Close')

def create_lstm_data_sel(data, seq_len=SEQ_LEN):
    X_, y_ = [], []
    for i in range(len(data) - seq_len):
        X_.append(data[i:i + seq_len])
        y_.append(data[i + seq_len][CLOSE_IDX2])
    return np.array(X_), np.array(y_)

X2, y2 = create_lstm_data_sel(scaled2)

model_final = Sequential([
    LSTM(64, activation='relu', input_shape=(X2.shape[1], X2.shape[2])),
    Dense(1)
], name='LSTM_HMM_Selected')
model_final.compile(optimizer='adam', loss='mse')
es2 = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)
model_final.fit(X2, y2, epochs=50, batch_size=16, verbose=1, callbacks=[es2])

# -- predictions ---------------------------------------------------------------
preds2   = model_final.predict(X2, verbose=0)
pf2      = np.zeros((len(preds2), scaled2.shape[1]))
pf2[:, CLOSE_IDX2] = preds2[:, 0]
restored2 = scaler2.inverse_transform(pf2)[:, CLOSE_IDX2]
real2     = df_sel['Close'].iloc[SEQ_LEN:SEQ_LEN + len(restored2)].values
dates2    = df_sel.index[SEQ_LEN:SEQ_LEN + len(restored2)]

# -- metrics -------------------------------------------------------------------
mae2  = mean_absolute_error(real2, restored2)
mse2  = mean_squared_error(real2, restored2)
rmse2 = np.sqrt(mse2)
mape2 = mean_absolute_percentage_error(real2, restored2) * 100

print('=' * 50)
print('   Final LSTM Model - Selected Features')
print('=' * 50)
print(f'  MAE  : {mae2:.4f}')
print(f'  MSE  : {mse2:.4f}')
print(f'  RMSE : {rmse2:.4f}')
print(f'  MAPE : {mape2:.2f} %')
print('=' * 50)

# -- next day ------------------------------------------------------------------
last_seq2   = np.expand_dims(scaled2[-SEQ_LEN:], axis=0)
next2_s     = model_final.predict(last_seq2, verbose=0)
dummy2      = np.zeros((1, scaled2.shape[1]))
dummy2[0, CLOSE_IDX2] = next2_s[0, 0]
next_day2   = scaler2.inverse_transform(dummy2)[0, CLOSE_IDX2]
print(f'\nPredicted next-day Close (final model): {next_day2:.2f} TRY')

## Cell 17 — Final Model Plots

In [ ]:
# -- full history --------------------------------------------------------------
plt.figure(figsize=(14, 6))
plt.plot(dates2, real2,     label='Actual Close', linewidth=1.5)
plt.plot(dates2, restored2, label='Prediction (Selected Features)', linewidth=1.5, alpha=0.8)
plt.title(f'{SYMBOL} - Actual vs Predicted (Final LSTM Model)')
plt.xlabel('Date')
plt.ylabel('Price (TRY)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# -- last 100 days -------------------------------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(dates2[-100:], real2[-100:],     marker='o', label='Actual Close', linewidth=2)
plt.plot(dates2[-100:], restored2[-100:], marker='x', linestyle='--',
         label='Prediction', linewidth=2)
plt.title(f'{SYMBOL} - Last 100 Days LSTM Forecast (Final Model)')
plt.xlabel('Date')
plt.ylabel('Price (TRY)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# -- last 200 days overlay -----------------------------------------------------
plt.figure(figsize=(12, 6))
plt.plot(dates2[-200:], real2[-200:],     color='black',  linewidth=2, label='Actual Close')
plt.plot(dates2[-200:], restored2[-200:], color='orange', linewidth=1.8, linestyle='-',
         label='LSTM Prediction')
plt.title(f'{SYMBOL} - Last 200 Days Prediction Performance')
plt.xlabel('Date')
plt.ylabel('Price (TRY)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## Cell 18 — Crossover Trading Strategy (200-Day Backtest)

**Strategy:** Detect crossovers between the actual price and LSTM prediction.
- Actual crosses **above** prediction → **BUY** (prediction under-prices the asset)
- Actual crosses **below** prediction → **SELL**

Starting capital: 100,000 TRY

In [ ]:
close_bt   = real2[-200:]
pred_bt    = restored2[-200:]

capital  = 100_000.0
cash     = capital
position = 0.0
trade_log = []

for i in range(len(close_bt) - 1):
    diff_now  = close_bt[i]     - pred_bt[i]
    diff_next = close_bt[i + 1] - pred_bt[i + 1]

    if diff_now * diff_next < 0:   # crossover
        alpha       = abs(diff_now) / (abs(diff_now) + abs(diff_next) + 1e-12)
        cross_idx   = i + alpha
        cross_price = float(close_bt[i] + alpha * (close_bt[i + 1] - close_bt[i]))

        if diff_now < 0:            # actual crosses above prediction -> BUY
            if cash > 0:
                position = cash / cross_price
                cash     = 0.0
                trade_log.append(('Buy',  cross_idx, cross_price))
        else:                       # actual crosses below prediction -> SELL
            if position > 0:
                cash     = position * cross_price
                position = 0.0
                trade_log.append(('Sell', cross_idx, cross_price))

final_value  = cash if position == 0 else position * float(close_bt[-1])
total_profit = final_value - capital

print(f'Starting Capital       : {capital:,.2f} TRY')
print(f'Final Portfolio Value  : {final_value:,.2f} TRY')
print(f'Total Profit / Loss    : {total_profit:+,.2f} TRY')
print(f'Return                 : {total_profit / capital * 100:+.2f} %')

# -- plot with signals ---------------------------------------------------------
plt.figure(figsize=(14, 6))
plt.plot(close_bt, label='Actual Close',      color='black')
plt.plot(pred_bt,  label='LSTM Prediction',   color='blue', linestyle='--')

buy_plotted = sell_plotted = False
for action, idx, price in trade_log:
    if action == 'Buy':
        lbl = 'Buy' if not buy_plotted else '_nolegend_'
        plt.scatter(idx, price, marker='^', color='green', s=120, label=lbl, zorder=5)
        buy_plotted = True
    else:
        lbl = 'Sell' if not sell_plotted else '_nolegend_'
        plt.scatter(idx, price, marker='v', color='red',   s=120, label=lbl, zorder=5)
        sell_plotted = True

plt.title(f'{SYMBOL} - Crossover Trading Strategy (200-Day Backtest)')
plt.xlabel('Day Index')
plt.ylabel('Price (TRY)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 19 — Trade Summary

In [ ]:
trade_pairs = []
for i in range(0, len(trade_log) - 1, 2):
    if trade_log[i][0] == 'Buy' and trade_log[i + 1][0] == 'Sell':
        _, buy_idx,  buy_price  = trade_log[i]
        _, sell_idx, sell_price = trade_log[i + 1]

        qty            = capital / buy_price
        total_buy_cost = qty * buy_price
        total_sell_rev = qty * sell_price
        gain           = total_sell_rev - total_buy_cost
        profit_pct     = (gain / total_buy_cost) * 100

        trade_pairs.append({
            'Buy Price':          round(float(buy_price),  2),
            'Sell Price':         round(float(sell_price), 2),
            'Buy Quantity':       round(float(qty),            2),
            'Total Buy Cost':     round(float(total_buy_cost), 2),
            'Total Sell Revenue': round(float(total_sell_rev), 2),
            'Profit (TRY)':       round(float(gain),           2),
            'Profit (%)':         round(float(profit_pct),     2),
            'Successful?':        '[OK]' if gain > 0 else '[X]',
        })

summary_df = pd.DataFrame(trade_pairs)

pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Trade Summary (Buy-Sell Pairs):')
print(summary_df.to_string(index=False))

if len(summary_df):
    total_trades    = len(summary_df)
    successful      = (summary_df['Successful?'] == '[OK]').sum()
    success_rate    = successful / total_trades * 100
    total_pnl       = summary_df['Profit (TRY)'].sum()

    print(f'\nTotal Trades     : {total_trades}')
    print(f'Successful       : {successful}')
    print(f'Success Rate     : {success_rate:.2f} %')
    print(f'Cumulative P&L   : {total_pnl:+,.2f} TRY')